# TPP1–telomerase: the structural test this sandbox could not run

**Run all. CPU runtime. ~2 minutes.** No GPU needed.

This notebook is a thin wrapper around **`colab/tpp1_colab.py`**, which is the single source of
truth and lives in the repository. That matters: the notebook does not contain its own copy of
the logic, so it cannot drift out of sync with the tested script.

### What it answers

Does the TEN-domain anchor motif **PLYQ (174–177)** actually contact the DNA once the substrate is
long enough to reach it?

All local structures (7TRE / 7TRD / 7TRF) carry only a **7-nucleotide** DNA — its 3′ end sits on
the catalytic aspartates and its 5′ end on the anchor strand R756, stopping two nucleotides short
of PLYQ. Extrapolating the 5′ exit direction put Tyr176 at **6.8 Å**, but that was an inference.
**7QXS / 7QXA (30 nt)** settle it.

### Why it is not already done

`files.rcsb.org` is blocked in the environment where this analysis was built. Colab has internet.
That is the only reason this notebook exists — **not compute.**

In [ ]:
# --- 1. get the code -------------------------------------------------------
# If you uploaded this notebook on its own, clone the repo. If you opened it
# from the repo, this is a no-op.

import os, subprocess, sys

REPO_URL = 'https://github.com/Mr-Nobody-Anonymous/telomerase-allosteric-switch.git'
REPO_DIR = 'telomerase-allosteric-switch'

if not os.path.exists('colab/tpp1_colab.py'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print('working dir:', os.getcwd())
print('runner present:', os.path.exists('colab/tpp1_colab.py'))

In [ ]:
# --- 2. install dependencies ----------------------------------------------
!pip -q install numpy scipy gemmi

import numpy, scipy, gemmi
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| gemmi', gemmi.__version__)

## 3. Self-test — run this first

Offline. Checks the script against the three bundled CIFs where the answer is already known
(PLYQ = 11.11 Å in 7TRE, 11.94 Å in 7TRD/7TRF), checks that TPP1 is found by **contact** and that
the histones in 7TRF are **not** mistaken for it, and exercises every verdict branch and the
RAP-state comparison with injected known motions.

**If this fails, stop. Do not run the download stage.** A failure here means the measurement
code is broken, and any result from stage 4 would be meaningless.

In [ ]:
!python colab/tpp1_colab.py --selftest

## 4. The real run — downloads 8 structures and measures

This fetches 7QXS, 7QXA, 9SHY, 9SHZ, 9SI0, plus the three local references, then measures:

* PLYQ / Y176 / TRAP-wedge (H752–H754) / anchor strand / IFD-TRAP / catalytic aspartates → DNA
* **every TEN-domain residue that contacts DNA**, per structure
* PLYQ distance **per nucleotide**, to see where along the DNA the anchor site sits
* TPP1 → DNA and TPP1 → PLYQ
* the three 2026 RAP states superposed on the RT ring, to see how far TEN and IFD-TRAP move
  **relative to the rigid ring** — not per-domain RMSD, which would absorb the motion

In [ ]:
!python colab/tpp1_colab.py

## 5. The report

Also written to `colab_out/report.md`. Paste this block back into the conversation — the verdict
line plus the distance table is enough.

In [ ]:
from pathlib import Path
p = Path('colab_out/report.md')
print(p.read_text() if p.exists() else 'No report produced — stage 4 must have failed.')

In [ ]:
# --- 6. per-nucleotide PLYQ profile (where along the DNA the anchor site sits) ---
import json
d = json.loads(Path('colab_out/results.json').read_text())
for s in d['structures']:
    prof = s.get('plyq_per_nt')
    if not prof:
        continue
    print(f"{s['pdb']} ({s['n_dna_nt']} nt, residues {s.get('dna_range')})")
    for row in prof:
        bar = '#' * max(0, int(30 - row['d'] * 2))
        print(f"   nt {row['nt']:>3d}  {row['d']:6.2f} A  {bar}")
    print()

## 7. What this cannot do

**It cannot settle the mechanism.** Structures show what is *reachable*, not which route TPP1
uses. That needs the bench measurements in `results/solve/PREREGISTRATION_WP14.md`:

1. **A on untreated enzyme** — one reaction; sets the operating point.
2. **w with a 95% CI** — active fraction; decides whether the feasibility gate is interpretable.
3. **tau across ≥3 purifications** — decides whether the study can be sized at all.

### On the GPU

You have Colab Pro VRAM. Where it would help, honestly:

| task | GPU value | should you? |
|---|---|---|
| this notebook | none | CPU runtime, ~2 min |
| the kinetic algebra (wp10–wp15) | none | already done |
| long-timescale MD of TEN/TPP1 | real, days–weeks | **no** — cannot identify a rate the observables don't constrain |
| ColabFold on a TERT mutant | modest, minutes | maybe, if a mutant has no structure |
| cryo-EM processing | large | only if you collect data |

The repository already tried 3.0 ps MD and retired an ANM result. The open question is
**experimental identifiability**, not sampling.